<a href="https://colab.research.google.com/github/mykisaacs/MyMassivelyParallelCode/blob/main/cpp/CUDA_Overview_Hello_world%2BVectorAddition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Running CUDA C++ (`.cu`) Code in Colab

Google Colab provides GPUs and pre-installed CUDA toolkits, making it an excellent environment for developing and running CUDA C++ applications. While you mentioned `.cpp`, for CUDA code, the standard extension is `.cu`.

Let's go through the steps:
1.  **Verify CUDA Installation**
2.  **Create a CUDA C++ Source File (`.cu`)**
3.  **Compile the CUDA Code using `nvcc`**
4.  **Run the Executable**

### 1. Verify CUDA Installation

First, let's check if `nvcc` (the NVIDIA CUDA compiler) is available and see its version. Colab instances usually come with CUDA pre-installed.

In [ ]:
import subprocess

try:
    # Check nvcc version
    result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True, check=True)
    print(result.stdout)
except FileNotFoundError:
    print("nvcc not found. CUDA Toolkit might not be installed or not in PATH.")
except subprocess.CalledProcessError as e:
    print(f"Error checking nvcc version: {e}\n{e.stderr}")


nvcc not found. CUDA Toolkit might not be installed or not in PATH.


### 2. Create a CUDA C++ Source File (`.cu`)

We'll create a simple CUDA C++ program named `hello_cuda.cu` that prints "Hello from CUDA!" from the GPU.

In [ ]:
%%writefile hello_cuda.cu
#include <iostream>

// CUDA kernel function
__global__ void helloFromGPU()
{
    printf("Hello from CUDA!\n");
}

int main()
{
    // Launch the kernel on the GPU
    // 1 block, 1 thread per block
    helloFromGPU<<<1, 1>>>();

    // Wait for the GPU to finish its work
    cudaDeviceSynchronize();

    // Print a message from the CPU
    std::cout << "Hello from CPU!" << std::endl;

    return 0;
}

Writing hello_cuda.cu


### 3. Compile the CUDA Code using `nvcc`

Now, we'll use the `nvcc` compiler to compile our `hello_cuda.cu` file into an executable. We'll name the output executable `hello_cuda`.

In [ ]:
!nvcc hello_cuda.cu -o hello_cuda

/bin/bash: line 1: nvcc: command not found


### 4. Run the Executable

Finally, execute the compiled program.

In [ ]:
!./hello_cuda

/bin/bash: line 1: ./hello_cuda: No such file or directory


### 5. Basic Code Example: Vector Addition

Below is a simple structural C++ implementation demonstrating how threads calculate their global data indexes independently.

In [ ]:
%%writefile vector_addition.cu

#include <iostream>
#include <cuda_runtime.h>

// __global__ keyword indicates the function runs on the GPU but is called from the CPU
__global__ void vectorAdd(const float* A, const float* B, float* C, int N) {
  // Built-in variables calculate a unique global ID for each thread
  int idx = blockIdx.x * blockDim.x + threadIdx.x;

  // Check boundary condition to prevent out-of-bounds memory access
  if (idx < N) {
    C[idx] = A[idx] + B[idx];
  }
}

int main() {
  int N = 1 << 20; // 1M elements
  size_t size = N * sizeof(float) ;

  // 1. Allocate Host Memory
  float *h_A = (float*)malloc(size);
  float *h_B = (float*)malloc(size);
  float *h_C = (float*)malloc(size);

  // Initialize array values
  for (int i = 0k i<N; i++) {
    h_A[i] = 1.0f ;
    h_B[i] = 2.0f ;
  }

  // 2. Allocate Device (GPU) Memory
  float *d_A, *d_B, *d_C ;
  cudaMalloc(&d_A, size) ;
  cudaMalloc(&d_B, size);
  cudaMalloc(&d_C, size);

  // 3. Copy Data from Host to Device
  cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice) ;
  cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice) ;

  // 4. Configure Execution parameters (256 threads per block)
  int threadsPerBlock = 256 ;
  int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock ;


}